# A Motivating Example: the Bigram Language Model

## Why is such a simple model interesting?

## The Bi-Gram Model

**Alice was beginning to get very tired of sitting by her sister on the bank.**

$$
P(A \cap B \cap C) = P(A|B \cap C) P(B|C) P(C)
$$

$$
\hat{P}(w_t | w_{t-1}) = \frac{C(w_{t-1}, w_t)}{C(w_{t-1})}
$$

## Using the model: scoring and generating text

## Limitations: why counting is not enough

## The Bi-Gram Model as a Classification Problem

## The Bi-Gram Model as a Logistic Regression

$$
\hat{y}_k = \text{softmax}(o)_k = \frac{e^{o_k}}{\sum_{j=1}^V e^{o_j}}, \qquad k = 1, \ldots, V
$$

## From counting to learning: the hidden layer

$$
\hat{y} = \text{softmax}\big( \varphi(x W_h + b_h) W_o + b_o \big)
$$

## A short history

## Implementation

In [ ]:
import re
from collections import Counter

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import gutenberg

nltk.download('gutenberg', quiet=True)

alice_text = gutenberg.raw('carroll-alice.txt')

# A very simple tokenizer: lower-case words (letters and apostrophes only)
tokens = re.findall(r"[a-z']+", alice_text.lower())

print(f"Number of tokens: {len(tokens)}")
print(f"Vocabulary size: {len(set(tokens))}")
print("First tokens:", tokens[:12])

In [ ]:
toy_sentence = "alice was beginning to get very tired".split()

toy_vocab = sorted(set(toy_sentence))
toy_index = {word: i for i, word in enumerate(toy_vocab)}

# Count how often each word (row) is followed by each other word (column)
toy_counts = np.zeros((len(toy_vocab), len(toy_vocab)))

for first_word, second_word in zip(toy_sentence[:-1], toy_sentence[1:]):
    toy_counts[toy_index[first_word], toy_index[second_word]] += 1

# Divide each row by its total (rows without any bigram stay zero)
row_totals = toy_counts.sum(axis=1, keepdims=True)
toy_probs = np.divide(toy_counts, row_totals, out=np.zeros_like(toy_counts), where=row_totals > 0)

pd.DataFrame(toy_probs, index=toy_vocab, columns=toy_vocab)

### The whole book

In [ ]:
bigram_counts = Counter(zip(tokens[:-1], tokens[1:]))
first_counts = Counter(tokens[:-1])
unigram_counts = Counter(tokens)

def next_word_probs(word, top=5):
    """The `top` most likely next words and their estimated probabilities"""
    candidates = [(second, count / first_counts[word])
                  for (first, second), count in bigram_counts.items() if first == word]
    return sorted(candidates, key=lambda pair: -pair[1])[:top]

for word in ["alice", "the", "said"]:
    print(f"After '{word}':", [(w, round(p, 3)) for w, p in next_word_probs(word)])

### Scoring sentences

In [ ]:
def sentence_log_prob(sentence):
    words = sentence.lower().split()
    log_prob = np.log(unigram_counts[words[0]] / len(tokens))

    for first, second in zip(words[:-1], words[1:]):
        if bigram_counts[(first, second)] == 0:
            return -np.inf     # unseen bigram: probability zero
        log_prob += np.log(bigram_counts[(first, second)] / first_counts[first])

    return log_prob

print("Natural order: ", sentence_log_prob("alice was beginning to get very tired"))
print("Reversed order:", sentence_log_prob("tired very get to beginning was alice"))

### Generating text

In [ ]:
rng = np.random.default_rng(1)

def sample_sentence(start_word, length=12):
    words = [start_word]
    for _ in range(length - 1):
        candidates = [(second, count) for (first, second), count in bigram_counts.items()
                      if first == words[-1]]
        if not candidates:
            break
        next_words, counts = zip(*candidates)
        words.append(rng.choice(next_words, p=np.array(counts) / sum(counts)))
    return " ".join(words)

for _ in range(3):
    print(sample_sentence("alice"))

### Sparsity